# Point Transformer for 3D Part Segmentation

3D Part Segmentation on ShapeNet: Point transformer architecture for dense semantic 3D part labeling. This notebook implements the approach with `PointTransformerConv` inside a `K3PointTransformerSeg` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `PointTransformerConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Point Transformer for 3D Part Segmentation"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Point Transformer Segmentation Model
class K3PointTransformerSeg(keras.Model):
    def __init__(self, in_channels=3, num_classes=16):
        super().__init__()
        self.conv1 = k3_layers.PointTransformerConv(in_channels, 32)
        self.conv2 = k3_layers.PointTransformerConv(32, 64)
        self.lin = layers.Dense(num_classes)

    def call(self, pos, edge_index):
        x = ops.relu(self.conv1(pos, pos, edge_index))
        x = ops.relu(self.conv2(x, pos, edge_index))
        return self.lin(x)

k3_model = K3PointTransformerSeg(in_channels=3, num_classes=16)

# 2. Forward pass test
num_points = 64
dummy_pos = keras.random.normal((num_points, 3))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")

out = k3_model(dummy_pos, dummy_edges)
print(f"PointTransformer segmentation output shape: {out.shape}")

print("\n✓ K3-Node Point Transformer Segmentation execution completed successfully!")